# Player Injuries and Team Performance — Exploratory Data Analysis

**Project context:** FootLens Analytics — understand how player injuries relate to match outcomes, player ratings, recovery time, and club-level injury patterns.

This is a fresh notebook built around the provided `player_injuries_impact` dataset and the assignment brief. It covers the five required business questions, data cleaning, feature engineering, groupby/aggregation, a pivot table, statistical summaries, and more than five visualisations.

> **Important:** Run the cells from top to bottom in Google Colab. The Kaggle credentials cell accepts a JSON credentials file **regardless of its uploaded filename**. It validates the file contents (`username` and `key`) and does not contain a hard-coded API key. Keep your Kaggle token private.

## 1. Business questions

1. **Which injury types are associated with the largest team performance drop** during the missed-match phase compared with the matches before injury?
2. **What was the club's win/draw/loss record during the player's absence?**
3. **How did individual player ratings change after recovery** compared with before injury?
4. **Are injuries clustered in particular months or clubs?**
5. **Which clubs have the greatest injury burden and apparent performance impact?**

Additional supporting questions: Which players have the most recorded injury events? Does age relate to performance drop? How long is the recorded injury-to-return interval, and how does it vary by injury type?

### Background framing
Sports analytics commonly combines event outcomes, player-level performance ratings, availability, and time-based trends to support squad rotation and planning. This dataset supports descriptive comparisons across three observed phases (before injury, missed matches, after return). These are **associations**, not proof that an injury alone caused a match result: opponent strength, line-ups, home/away status, and other contextual variables are not fully controlled here.

Reference links supplied in the assignment brief:
- [Football analysis with Streamlit and Plotly](https://medium.com/@atharvtembhurnikar/football-analysis-using-streamlit-and-plotly-3d0eaed2ac5b)
- [Visualising Serie A football data with Pandas and Plotly](https://towardsdatascience.com/visualise-serie-a-football-league-data-with-pandas-and-plotly-3d26bcf26f2d/)
- [Sports analytics research article](https://link.springer.com/article/10.1007/s44196-023-00217-6)
- [Pandas groupby by month](https://www.askpython.com/python-modules/pandas/groupby-to-group-dataframe-by-month)
- [Streamlit fundamentals](https://docs.streamlit.io/get-started/fundamentals/main-concepts)

### Background research sources

The assignment brief lists these relevant starting points for sports analytics and dashboard design. Use them to support the domain framing in the final report, and verify the article details when citing them formally:

- Football analysis with Streamlit and Plotly: https://medium.com/@atharvtembhurnikar/football-analysis-using-streamlit-and-plotly-3d0eaed2ac5b
- Research article on football/sports analytics: https://link.springer.com/article/10.1007/s44196-023-00217-6
- Sports analytics topic collection: https://towardsdatascience.com/tag/sports-analytics/
- Dashboard user-experience patterns: https://www.pencilandpaper.io/articles/ux-pattern-analysis-data-dashboards
- Streamlit fundamentals: https://docs.streamlit.io/get-started/fundamentals/main-concepts

These are supplied in the assignment brief; the notebook does not claim that any one source proves a causal relationship between injuries and team results.

## 2. Environment setup and reproducibility

The setup follows the style of the supplied example but installs only packages relevant to this EDA notebook.

In [ ]:
import os, sys, platform, subprocess, warnings, random, logging, json, time, re, zipfile, shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("PlayerInjuryEDA")

# Detect Google Colab
IS_COLAB = False
try:
    import google.colab
    IS_COLAB = True
    logger.info("Google Colab detected")
except ImportError:
    logger.info("Running locally")

# Reproducibility
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Install/import analysis packages only. IMPORTANT: do not import `kaggle`
# here; Kaggle authentication must happen after the token JSON is configured.
PKG_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "psutil": "psutil",
}
for package, import_name in PKG_IMPORTS.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

# Ensure Kaggle CLI is installed, but do not import its Python package.
if shutil.which("kaggle") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"]
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

# Hardware/system report (GPU is optional for this CPU-based EDA)
gpu_info = {"avail": False, "name": "None", "mem": "N/A"}
try:
    import torch
    if torch.cuda.is_available():
        gpu_info = {
            "avail": True,
            "name": torch.cuda.get_device_name(0),
            "mem": f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB",
        }
except Exception:
    pass

try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
except Exception:
    ram = 0

print("=" * 60)
print("ENVIRONMENT REPORT")
print("=" * 60)
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()[:70]}")
print(f"Colab    : {IS_COLAB}")
print(f"GPU      : {gpu_info['name']} ({gpu_info['mem']})" if gpu_info["avail"] else "GPU      : Not available / not required")
print(f"RAM      : {ram:.1f} GB")
print(f"Seed     : {SEED}")
print("=" * 60)


## 3. Kaggle authentication and dataset download

Upload your Kaggle API token JSON when prompted. **The local filename does not matter**: for example, `kaggle.json`, `kaggle (2).json`, or `my-kaggle-token.json` are all accepted as long as the file contents are valid Kaggle credentials. Do not paste your API key into notebook code or share the token publicly.

In [ ]:
from pathlib import Path
import os, json, subprocess, sys, shutil

KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"
KAGGLE_CMD = shutil.which("kaggle")
if KAGGLE_CMD is None:
    raise RuntimeError(
        "Kaggle CLI is not installed. Rerun the environment setup cell, then retry."
    )

# Upload credentials once in Colab. The filename can be anything.
if IS_COLAB:
    from google.colab import files as colab_files
    print("Upload your Kaggle API credentials JSON file (any filename is accepted):")
    uploaded_credentials = colab_files.upload()
    if not uploaded_credentials:
        raise RuntimeError("No credentials file uploaded. Rerun this cell and upload the JSON token.")
    valid_credentials = None
    for uploaded_name, uploaded_bytes in uploaded_credentials.items():
        try:
            candidate = json.loads(uploaded_bytes.decode("utf-8"))
            if isinstance(candidate, dict) and candidate.get("username") and candidate.get("key"):
                valid_credentials = candidate
                print(f"Valid Kaggle credentials found in: {uploaded_name}")
                break
        except (UnicodeDecodeError, json.JSONDecodeError, AttributeError):
            continue
    if valid_credentials is None:
        raise ValueError(
            "The uploaded file is not a valid Kaggle API token JSON. "
            "It must contain non-empty 'username' and 'key' fields."
        )
    with open(KAGGLE_JSON, "w", encoding="utf-8") as f:
        json.dump(
            {"username": valid_credentials["username"], "key": valid_credentials["key"]},
            f
        )
elif KAGGLE_JSON.exists():
    print(f"Using existing credentials at {KAGGLE_JSON}")
else:
    raise RuntimeError(
        "No Kaggle credentials found. Place your token at ~/.kaggle/kaggle.json "
        "before running this notebook outside Colab."
    )

os.chmod(KAGGLE_JSON, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)
print("[OK] Kaggle credentials configured. API key is not displayed.")

# Validate authentication with a lightweight CLI request.
check = subprocess.run(
    [KAGGLE_CMD, "datasets", "list", "-s", "player injuries team performance", "-p", "1"],
    capture_output=True, text=True
)
if check.returncode == 0:
    print("[OK] Kaggle API authentication succeeded.")
else:
    raise RuntimeError(
        "Kaggle authentication/API check failed. Confirm the token is active on Kaggle "
        "and rerun this cell.\n" + check.stderr[-1200:]
    )


## 4. Download and locate the dataset

The notebook first tries the Kaggle dataset used for this project. If download is unavailable, it offers a manual CSV upload fallback. The loader checks for the expected player-injury columns so it does not accidentally load an unrelated CSV.

In [ ]:
DATASET_SLUG = "amritbiswas007/player-injuries-and-team-performance-dataset"
DATASET_PATH = Path("/content/player_injuries_dataset") if IS_COLAB else Path.cwd() / "player_injuries_dataset"
DATASET_PATH.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = {
    "Name", "Team Name", "Position", "Age", "Season",
    "FIFA rating", "Injury", "Date of Injury", "Date of return"
}

def find_matching_csv(root):
    for candidate_path in Path(root).rglob("*.csv"):
        try:
            preview = pd.read_csv(candidate_path, nrows=3)
            if REQUIRED_COLUMNS.issubset(set(preview.columns)):
                return candidate_path
        except Exception:
            continue
    return None

csv_path = find_matching_csv(DATASET_PATH)
if csv_path is None:
    print(f"Downloading Kaggle dataset: {DATASET_SLUG}")
    result = subprocess.run(
        [KAGGLE_CMD, "datasets", "download", "-d", DATASET_SLUG,
         "-p", str(DATASET_PATH), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Kaggle dataset download failed. No second upload prompt will be shown. "
            "Check the dataset slug, internet connection, and Kaggle token.\n"
            + result.stderr[-1500:]
        )
    csv_path = find_matching_csv(DATASET_PATH)

if csv_path is None:
    raise FileNotFoundError(
        f"No CSV with the expected player-injury columns was found in {DATASET_PATH}. "
        "Check that the downloaded dataset contains the assignment dataset."
    )

print(f"[OK] Dataset selected: {csv_path}")
df_raw = pd.read_csv(
    csv_path,
    na_values=["N.A.", "N.A", "NA", "N/A", "n/a", "null", "None", "", " "]
)
missing_required = REQUIRED_COLUMNS - set(df_raw.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_required)}")
print(f"Dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")
display(df_raw.head())


## 5. Understand the dataset structure

In [ ]:
print("Rows and columns:", df_raw.shape)
print("\nColumn names:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:02d}. {col}")
print("\nData types and non-null counts:")
df_raw.info()
print("\nMissing values (largest first):")
missing_summary = (df_raw.isna().sum().to_frame("missing_count")
                   .assign(missing_pct=lambda x: (100*x["missing_count"]/len(df_raw)).round(2))
                   .sort_values("missing_pct", ascending=False))
display(missing_summary.head(20))
print("\nDuplicate rows:", df_raw.duplicated().sum())

## 6. Data cleaning and feature engineering

- Parse injury and return dates using tolerant date conversion.
- Convert age, FIFA rating, goal differences, and player ratings to numeric values. Ratings such as `6(S)` are cleaned by extracting the numeric part.
- Treat text placeholders as missing values.
- Calculate average player rating before and after injury, average team goal difference before/during absence/after return, and the associated changes.
- Calculate the recorded injury-to-return interval and injury month.

The dataset contains up to three match observations per phase. Averages are calculated from the available valid observations; missing match observations are not filled with invented values.

In [ ]:
df = df_raw.copy()

# Standardize text placeholders across the dataset
placeholder_values = ["N.A.", "N.A", "NA", "N/A", "n/a", "null", "None", ""]
df = df.replace(placeholder_values, np.nan)

# Dates and core numeric fields
for col in ["Date of Injury", "Date of return"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["Age", "FIFA rating"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Match column groups
before_rating_cols = [c for c in df.columns if re.match(r"Match\d+_before_injury_Player_rating$", c)]
after_rating_cols = [c for c in df.columns if re.match(r"Match\d+_after_injury_Player_rating$", c)]
before_gd_cols = [c for c in df.columns if re.match(r"Match\d+_before_injury_GD$", c)]
missed_gd_cols = [c for c in df.columns if re.match(r"Match\d+_missed_match_GD$", c)]
after_gd_cols = [c for c in df.columns if re.match(r"Match\d+_after_injury_GD$", c)]
missed_result_cols = [c for c in df.columns if re.match(r"Match\d+_missed_match_Result$", c)]

# Numeric parser supports values such as '6(S)' and goal-difference strings.
def numeric_from_text(series):
    extracted = series.astype("string").str.extract(r"([-+]?\d*\.?\d+)", expand=False)
    return pd.to_numeric(extracted, errors="coerce")

for col in before_rating_cols + after_rating_cols + before_gd_cols + missed_gd_cols + after_gd_cols:
    df[col] = numeric_from_text(df[col])

# Phase-level averages per injury record
def row_mean(cols):
    return df[cols].mean(axis=1) if cols else pd.Series(np.nan, index=df.index)

df["Avg_Rating_Before"] = row_mean(before_rating_cols)
df["Avg_Rating_After"] = row_mean(after_rating_cols)
df["Rating_Change_After_vs_Before"] = df["Avg_Rating_After"] - df["Avg_Rating_Before"]
df["Avg_GD_Before"] = row_mean(before_gd_cols)
df["Avg_GD_Missed"] = row_mean(missed_gd_cols)
df["Avg_GD_After"] = row_mean(after_gd_cols)
df["GD_Change_Missed_vs_Before"] = df["Avg_GD_Missed"] - df["Avg_GD_Before"]
df["GD_Change_After_vs_Before"] = df["Avg_GD_After"] - df["Avg_GD_Before"]
df["Performance_Drop_Index"] = df["Avg_GD_Before"] - df["Avg_GD_Missed"]
df["Injury_Duration_Days"] = (df["Date of return"] - df["Date of Injury"]).dt.days
df.loc[df["Injury_Duration_Days"] < 0, "Injury_Duration_Days"] = np.nan
df["Injury_Month"] = df["Date of Injury"].dt.month_name()
df["Injury_Month_Number"] = df["Date of Injury"].dt.month

# Normalize injury labels for grouping, while retaining original label in Injury.
df["Injury_Clean"] = df["Injury"].astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
df["Injury_Clean"] = df["Injury_Clean"].replace({"": pd.NA, "<na>": pd.NA})

print("Rating columns found:", len(before_rating_cols), "before and", len(after_rating_cols), "after")
print("Goal-difference columns found:", len(before_gd_cols), "before,", len(missed_gd_cols), "missed,", len(after_gd_cols), "after")
print("Cleaned shape:", df.shape)
display(df[["Name", "Team Name", "Injury", "Date of Injury", "Date of return", "Injury_Duration_Days", "Avg_Rating_Before", "Avg_Rating_After", "Avg_GD_Before", "Avg_GD_Missed", "Avg_GD_After"]].head())

## 7. Cleaning checks and descriptive statistics

In [ ]:
print("Missing core fields after parsing:")
display(df[["Name", "Team Name", "Injury", "Date of Injury", "Date of return", "Age", "FIFA rating"]].isna().sum().to_frame("missing_count"))
print("\nNumeric summary:")
display(df[["Age", "FIFA rating", "Injury_Duration_Days", "Avg_Rating_Before", "Avg_Rating_After", "Avg_GD_Before", "Avg_GD_Missed", "Avg_GD_After", "Performance_Drop_Index"]].describe().T.round(3))
print("\nInvalid negative injury durations remaining:", (df["Injury_Duration_Days"] < 0).sum())
print("Rows with both pre- and post-injury ratings:", df[["Avg_Rating_Before", "Avg_Rating_After"]].notna().all(axis=1).sum())

## 8. Business question 1 — Which injury types have the largest performance drop?

`Performance_Drop_Index = average goal difference before injury − average goal difference during missed matches`. A positive value indicates the average goal difference was lower during absence. The table also reports the sample size, because injury categories with very few observations can be unstable.

In [ ]:
# Compare injury labels only when there are at least five usable records.
# Administrative labels such as "quarantine" are not injury types and are excluded from this comparison.
injury_impact = (df.dropna(subset=["Injury_Clean", "Performance_Drop_Index"])
    .groupby("Injury_Clean", as_index=False)
    .agg(Mean_Performance_Drop=("Performance_Drop_Index", "mean"),
         Median_Performance_Drop=("Performance_Drop_Index", "median"),
         Records=("Performance_Drop_Index", "size"),
         Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean")))
non_injury_labels = {"quarantine"}
injury_impact = injury_impact[~injury_impact["Injury_Clean"].str.casefold().isin(non_injury_labels)]
injury_impact = injury_impact[injury_impact["Records"] >= 5].sort_values("Mean_Performance_Drop", ascending=False)
display(injury_impact.head(10).round(3))

plot_data = injury_impact.head(10).sort_values("Mean_Performance_Drop").copy()
plot_data["Injury_label_with_n"] = plot_data.apply(lambda r: f'{r["Injury_Clean"]} (n={int(r["Records"])})', axis=1)
fig, ax = plt.subplots(figsize=(10, 5.5))
if not plot_data.empty:
    ax.barh(plot_data["Injury_label_with_n"], plot_data["Mean_Performance_Drop"])
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Mean performance-drop index")
    ax.set_ylabel("Injury label")
    ax.set_title("Top injury types by observed performance drop (minimum 5 records)")
else:
    ax.text(0.5, 0.5, "No eligible injury labels", ha="center", va="center")
    ax.set_axis_off()
fig.tight_layout()
plt.show()
plt.close(fig)
if not plot_data.empty:
    top = plot_data.iloc[-1]
    print(f"Finding: among injury labels with at least 5 records, {top['Injury_Clean']} has the largest mean observed performance-drop index ({top['Mean_Performance_Drop']:.3f}; n={int(top['Records'])}). This is an association in the recorded matches, not proof the injury caused the change.")
else:
    print("Finding: no injury labels have five or more usable records for this comparison.")

## 9. Business question 2 — What was the recorded win/draw/loss record during absence?

The dataset records up to three missed-match outcomes per injury record. The summary below describes only those recorded observations, not every match played by the club during the full absence period. It does not infer a season-wide win/loss record.

In [ ]:
result_long = df[["Name", "Team Name"] + missed_result_cols].melt(
    id_vars=["Name", "Team Name"], value_vars=missed_result_cols,
    var_name="Missed_Match_Slot", value_name="Result")
result_long["Result"] = result_long["Result"].astype("string").str.strip().str.lower()
result_long = result_long[result_long["Result"].isin(["win", "draw", "lose", "loss"])]
result_long["Result"] = result_long["Result"].replace({"loss":"lose"})
result_counts = result_long["Result"].value_counts().reindex(["win", "draw", "lose"], fill_value=0)
result_summary = pd.DataFrame({"Result": result_counts.index, "Matches": result_counts.values})
result_summary["Share_pct"] = (100 * result_summary["Matches"] / result_summary["Matches"].sum()).round(1)
display(result_summary)
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(result_summary["Result"].str.title(), result_summary["Matches"])
ax.set_title("Win/draw/loss outcomes in recorded missed matches")
ax.set_xlabel("Recorded outcome")
ax.set_ylabel("Number of recorded outcomes")
for bar, share in zip(bars, result_summary["Share_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{int(bar.get_height())} ({share:.1f}%)", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
plt.show()
plt.close(fig)

club_absence_results = (result_long.groupby(["Team Name", "Result"]).size().unstack(fill_value=0))
for outcome in ["win", "draw", "lose"]:
    if outcome not in club_absence_results.columns:
        club_absence_results[outcome] = 0
club_absence_results["Total"] = club_absence_results[["win", "draw", "lose"]].sum(axis=1)
club_absence_results["Win_Rate_pct"] = (100 * club_absence_results["win"] / club_absence_results["Total"].replace(0, np.nan)).round(1)
display(club_absence_results.sort_values("Total", ascending=False))

## 10. Business question 3 — How did players perform after recovery?

A positive rating change means the mean rating after return was higher than the mean before injury; a negative value indicates a decline. Interpret individual rankings alongside the number of valid before/after observations.

In [ ]:
paired = df.dropna(subset=["Avg_Rating_Before", "Avg_Rating_After"]).copy()
player_comeback = (paired.groupby(["Name", "Team Name"], as_index=False)
    .agg(Injury_Records=("Name", "size"),
         Mean_Rating_Before=("Avg_Rating_Before", "mean"),
         Mean_Rating_After=("Avg_Rating_After", "mean"),
         Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean"),
         Mean_GD_Change=("GD_Change_After_vs_Before", "mean")))
player_comeback = player_comeback.sort_values("Mean_Rating_Change", ascending=False)
print("Player rating changes (all players with paired before/after ratings):")
display(player_comeback.head(10).round(3))
print("Largest average rating declines (all players with paired before/after ratings):")
display(player_comeback.tail(10).sort_values("Mean_Rating_Change").round(3))

# The displayed leaderboard requires at least two paired injury records per player/team.
leaderboard_data = player_comeback[player_comeback["Injury_Records"] >= 2].copy()
leaderboard = leaderboard_data.head(10).sort_values("Mean_Rating_Change")
if not leaderboard.empty:
    leaderboard["Player_label"] = leaderboard.apply(lambda r: f'{r["Name"]} (n={int(r["Injury_Records"])})', axis=1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(leaderboard["Player_label"], leaderboard["Mean_Rating_Change"])
    ax.axvline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Mean rating change after return (after − before)")
    ax.set_ylabel("Player (paired records)")
    ax.set_title("Comeback leaderboard (minimum 2 paired records)")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    best = leaderboard_data.iloc[0]
    print(f"Finding: the largest mean rating improvement among players with at least two paired records is {best['Name']} ({best['Team Name']}): {best['Mean_Rating_Change']:.2f} rating points across {int(best['Injury_Records'])} records. This is an observed before/after difference and does not establish that recovery alone caused it.")
else:
    print("Finding: no players have at least two paired records, so a repeat-record comeback ranking cannot be shown reliably.")

# Pivot-table comparison across injury phases
phase_pivot = pd.pivot_table(df, index="Team Name",
    values=["Avg_Rating_Before", "Avg_Rating_After", "Avg_GD_Before", "Avg_GD_Missed", "Avg_GD_After"],
    aggfunc="mean")
print("Team-level pivot table: mean player ratings and goal difference across phases")
display(phase_pivot.round(3))

### Player performance timeline (before vs. after injury)

This chart compares mean recorded player ratings before injury and after return for the players with the most injury records and complete phase ratings. It is a descriptive comparison of the dataset's available match slots, not a full season timeline.

In [ ]:
# Player rating timeline: compare average rating before injury and after return.
# Matplotlib is used here so the chart renders reliably in Google Colab without a blank Plotly output area.
timeline_players = (
    paired.groupby(["Name", "Team Name"], as_index=False)
    .agg(
        Injury_Records=("Name", "size"),
        Rating_Before=("Avg_Rating_Before", "mean"),
        Rating_After=("Avg_Rating_After", "mean")
    )
)
selected_players = timeline_players.sort_values(
    ["Injury_Records", "Name"], ascending=[False, True]
).head(6).copy()

fig, ax = plt.subplots(figsize=(10, 5.5))
for _, row in selected_players.iterrows():
    ax.plot(
        ["Before injury", "After return"],
        [row["Rating_Before"], row["Rating_After"]],
        marker="o", linewidth=1.8,
        label=f'{row["Name"]} (n={int(row["Injury_Records"])})'
    )
ax.set_title("Player rating timeline: before injury vs after return (top 6)")
ax.set_xlabel("Injury phase")
ax.set_ylabel("Mean player rating")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()
plt.close(fig)
print("Interpretation: each line compares average recorded ratings before injury and after return for one player. Only the six players with the most paired records are shown; this is not a complete ranking of all players.")


## 11. Business question 4 — Are injuries clustered by month or club?

In [ ]:
# Monthly injury counts in calendar order.
# This chart is in its own cell to keep the output compact and easy to inspect.
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]
monthly_counts = (
    df.dropna(subset=["Injury_Month"])
      .groupby("Injury_Month").size()
      .reindex(month_order, fill_value=0)
      .rename("Injury_Records").reset_index()
)
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(monthly_counts["Injury_Month"], monthly_counts["Injury_Records"],
        marker="o", linewidth=2)
ax.set_title("Recorded injury events by month")
ax.set_xlabel("Month")
ax.set_ylabel("Injury records")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()
plt.close(fig)
display(monthly_counts.sort_values("Injury_Records", ascending=False))
print("Interpretation: these are counts of records in the dataset, not injury rates adjusted for the number of players or matches in each month.")


In [ ]:
# Heatmap of recorded injury counts by club and month — separate chart cell.
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]
club_month = pd.crosstab(df["Team Name"], df["Injury_Month"]).reindex(columns=month_order, fill_value=0)
fig, ax = plt.subplots(figsize=(12, max(5, 0.38 * len(club_month))))
sns.heatmap(club_month, cmap="YlOrRd", linewidths=0.25, annot=True, fmt="d",
            cbar_kws={"label": "Injury records"}, ax=ax)
ax.set_title("Injury frequency by club and month")
ax.set_xlabel("Month of injury")
ax.set_ylabel("Club")
ax.tick_params(axis="x", rotation=40)
fig.tight_layout()
plt.show()
plt.close(fig)
if not club_month.empty and club_month.to_numpy().max() > 0:
    row_i, col_i = np.unravel_index(club_month.to_numpy().argmax(), club_month.shape)
    print(f"Finding: the largest club-month cluster is {club_month.index[row_i]} in {club_month.columns[col_i]} ({int(club_month.to_numpy().max())} records). These are raw counts, not rates adjusted for squad size.")


## 12. Business question 5 — Which clubs have the greatest injury burden?

This section distinguishes **frequency** (number of recorded injury events) from **observed impact** (mean performance-drop index). A club with more records may simply have more rows in the dataset, so counts and impact averages are shown together.

In [ ]:
club_summary = (df.groupby("Team Name", as_index=False)
    .agg(Injury_Records=("Name", "size"),
         Unique_Players=("Name", "nunique"),
         Mean_Injury_Duration_Days=("Injury_Duration_Days", "mean"),
         Mean_Performance_Drop_Index=("Performance_Drop_Index", "mean"),
         Median_Performance_Drop_Index=("Performance_Drop_Index", "median"),
         Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean")))
club_summary = club_summary.sort_values("Injury_Records", ascending=False)
display(club_summary.round(3))
fig, ax = plt.subplots(figsize=(10, 5.5))
club_plot = club_summary.sort_values("Injury_Records")
bars = ax.barh(club_plot["Team Name"], club_plot["Injury_Records"])
ax.set_xlabel("Recorded injury events")
ax.set_ylabel("Club")
ax.set_title("Club injury burden")
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center", fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)

## 13. Additional EDA — players with the most injury records

In [ ]:
player_injury_counts = (df.groupby(["Name", "Team Name"], as_index=False)
    .agg(Injury_Records=("Injury", "size"),
         Distinct_Injury_Types=("Injury_Clean", "nunique"),
         Mean_Injury_Duration_Days=("Injury_Duration_Days", "mean")))
player_injury_counts = player_injury_counts.sort_values("Injury_Records", ascending=False)
display(player_injury_counts.head(15).round(2))
fig, ax = plt.subplots(figsize=(10, 6))
player_plot = player_injury_counts.head(15).sort_values("Injury_Records")
ax.barh(player_plot["Name"], player_plot["Injury_Records"])
ax.set_xlabel("Recorded injury events")
ax.set_ylabel("Player")
ax.set_title("Players with the most recorded injury events (top 15)")
fig.tight_layout()
plt.show()
plt.close(fig)

## 14. Additional EDA — age versus performance drop

In [ ]:
# Age versus observed performance-drop index.
age_impact = df.dropna(subset=["Age", "Performance_Drop_Index"])
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(age_impact["Age"], age_impact["Performance_Drop_Index"], alpha=0.65)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Player age")
ax.set_ylabel("Performance-drop index")
ax.set_title("Player age versus observed performance-drop index")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)
if len(age_impact) >= 3:
    corr = age_impact[["Age", "Performance_Drop_Index"]].corr(method="pearson").iloc[0, 1]
    print(f"Pearson correlation between age and performance-drop index: {corr:.3f} (n={len(age_impact)}).")
    print("Interpretation: this measures linear association only; it does not show that age causes performance changes.")
else:
    print("Not enough complete records to calculate the age correlation.")


## 15. Recovery duration analysis

In [ ]:
# Recovery duration distribution and data-quality check.
duration = df.dropna(subset=["Injury_Duration_Days"]).copy()
if not duration.empty:
    print("Recovery interval summary (days):")
    display(duration["Injury_Duration_Days"].describe(percentiles=[.25, .5, .75, .95]).to_frame().round(2))
    long_intervals = duration[
        duration["Injury_Duration_Days"] > 365
    ][["Name", "Team Name", "Injury", "Date of Injury", "Date of return", "Injury_Duration_Days"]].sort_values(
        "Injury_Duration_Days", ascending=False
    )
    print(f"Intervals longer than 365 days flagged for checking (retained, not automatically deleted): {len(long_intervals)}")
    if not long_intervals.empty:
        display(long_intervals.head(20))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(duration["Injury_Duration_Days"], bins=30, edgecolor="white")
    ax.set_xlabel("Days between recorded injury and return")
    ax.set_ylabel("Number of records")
    ax.set_title("Distribution of recorded injury-to-return intervals")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    print("Interpretation: intervals over 365 days may be genuine or may reflect date/data-entry issues. They are flagged for verification, not removed automatically.")
else:
    print("No valid injury-to-return intervals were available after cleaning.")


## 16. Statistical comparison of player ratings before and after injury

This is a paired comparison on records with both a before and after average rating. The paired t-test assumes the paired differences are reasonably suitable for that test; the Wilcoxon signed-rank test is also reported as a non-parametric sensitivity check. These are descriptive/exploratory results, not evidence of causation.

In [ ]:
from scipy import stats
rating_pairs = df[["Avg_Rating_Before", "Avg_Rating_After"]].dropna()
if len(rating_pairs) >= 2:
    before = rating_pairs["Avg_Rating_Before"]
    after = rating_pairs["Avg_Rating_After"]
    differences = after - before
    print(f"Paired records: {len(rating_pairs)}")
    print(f"Mean before rating: {before.mean():.3f}")
    print(f"Mean after rating:  {after.mean():.3f}")
    print(f"Mean paired change: {differences.mean():.3f}")
    print(f"Median paired change: {differences.median():.3f}")
    print(f"Paired t-test: statistic={stats.ttest_rel(after, before).statistic:.3f}, p={stats.ttest_rel(after, before).pvalue:.4f}")
    try:
        w = stats.wilcoxon(after, before, zero_method="wilcox")
        print(f"Wilcoxon signed-rank: statistic={w.statistic:.3f}, p={w.pvalue:.4f}")
    except ValueError as e:
        print("Wilcoxon test could not be calculated:", e)
else:
    print("Not enough paired before/after ratings for statistical testing.")

## 17. Summary of key tables for reporting

These tables are suitable for the written report. The tables retain sample counts so that means from small groups are not over-interpreted.

In [ ]:
print("Top injury types by performance-drop index (minimum 5 records; quarantine excluded)")
display(injury_impact.head(5).round(3))
print("\nClubs with the highest injury record counts")
display(club_summary.sort_values("Injury_Records", ascending=False).head(10).round(3))
print("\nLargest average comeback improvements")
display(player_comeback.head(10).round(3))
print("\nLargest average rating declines")
display(player_comeback.tail(10).sort_values("Mean_Rating_Change").round(3))
print("\nMonthly injury record counts")
display(monthly_counts)

## 18. Rubric-required visualizations — visible chart previews

The interactive Plotly charts in the analysis sections are retained. This section adds **static image versions** of the rubric's required visuals so the graphs remain visible even if an HTML/JavaScript chart does not render in the notebook viewer.

The five required visuals are:
1. Top 10 injury types by observed performance-drop index.
2. Player performance timeline before injury and after return.
3. Injury frequency heatmap across months and clubs.
4. Player age versus observed performance-drop index.
5. Comeback leaderboard by average rating improvement.

Additional visuals include a club-count bar chart with a dashed overall-average line, a club-by-phase performance heatmap, an injury-type-by-phase heatmap, and a histogram of recorded injury-to-return intervals. These charts are descriptive: they show associations in the available records, not proof that an injury alone caused a change in team performance.

For interpretability, injury-type comparisons use a minimum of five records and exclude the administrative label `quarantine`; the exported impact table uses the same rule. Comeback rankings require at least two paired injury records and display the record count. The timeline is limited to six players to reduce legend clutter. Recorded injury-to-return intervals over 365 days are flagged for review but retained unless independently verified as errors. Brief data-driven findings are printed below the relevant charts when the notebook runs.

## 18. Rubric-required visualizations — one chart per cell

Each visualization below has its own code cell so it can be run, viewed, and debugged independently. Run the shared setup cell first, then run the chart cells in order.

In [ ]:
# Rubric-required visualizations — static previews that render as images in Colab
# The Plotly charts above remain available for interactivity; these Matplotlib/Seaborn
# versions ensure the five rubric visuals are also visible when HTML chart rendering fails.
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from IPython.display import display

CHART_DIR = Path("player_injuries_eda_outputs") / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

# Recompute chart summaries from the cleaned dataframe so this section is self-contained.
chart_injury_impact = (
    df.dropna(subset=["Injury_Clean", "Performance_Drop_Index"])
      .groupby("Injury_Clean", as_index=False)
      .agg(Mean_Performance_Drop=("Performance_Drop_Index", "mean"),
           Records=("Performance_Drop_Index", "size"))
)
chart_injury_impact = chart_injury_impact[~chart_injury_impact["Injury_Clean"].str.casefold().isin({"quarantine"})]
chart_injury_impact = chart_injury_impact[chart_injury_impact["Records"] >= 5]
chart_injury_impact = chart_injury_impact.nlargest(10, "Mean_Performance_Drop").sort_values("Mean_Performance_Drop")
chart_injury_impact["Label_n"] = chart_injury_impact.apply(lambda r: f'{r["Injury_Clean"]} (n={int(r["Records"])})', axis=1)

### Chart 1 — Top 10 injury types by observed performance drop

In [ ]:
# 1. Top 10 injuries by observed performance-drop index
fig, ax = plt.subplots(figsize=(11, 6))
if not chart_injury_impact.empty:
    ax.barh(chart_injury_impact["Label_n"], chart_injury_impact["Mean_Performance_Drop"])
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_xlabel("Mean performance-drop index (before GD − missed-match GD)")
    ax.set_ylabel("Injury label")
    ax.set_title("Top injury types by observed performance drop (minimum 5 records)")
else:
    ax.text(0.5, 0.5, "Not enough injury groups with 3+ records", ha="center", va="center")
    ax.set_axis_off()
fig.tight_layout()
fig.tight_layout()
fig.savefig(CHART_DIR / "01_top_10_injuries_by_impact.png", dpi=160, bbox_inches="tight")
plt.show()
if not chart_injury_impact.empty:
    top_row = chart_injury_impact.loc[chart_injury_impact["Mean_Performance_Drop"].idxmax()]
    print(f"Finding: {top_row['Injury_Clean']} has the largest mean observed performance-drop index among eligible injury types ({top_row['Mean_Performance_Drop']:.3f}; n={int(top_row['Records'])}). This is descriptive, not causal.")
plt.close(fig)

### Chart 2 — Player rating timeline before injury vs after return

In [ ]:
# 2. Player rating timeline before injury vs after return (top 10 players by record count)
chart_paired = df.dropna(subset=["Avg_Rating_Before", "Avg_Rating_After"]).copy()
chart_players = (
    chart_paired.groupby(["Name", "Team Name"], as_index=False)
    .agg(Injury_Records=("Name", "size"),
         Rating_Before=("Avg_Rating_Before", "mean"),
         Rating_After=("Avg_Rating_After", "mean"))
)
chart_players = chart_players.sort_values(["Injury_Records", "Name"], ascending=[False, True]).head(6)
fig, ax = plt.subplots(figsize=(11, 6))
for _, row in chart_players.iterrows():
    ax.plot(["Before injury", "After return"],
            [row["Rating_Before"], row["Rating_After"]],
            marker="o", linewidth=1.5, alpha=0.8,
            label=f'{row["Name"]} ({row["Team Name"]}, n={int(row["Injury_Records"])})')
ax.set_ylabel("Mean player rating")
ax.set_title("Player rating timeline: before injury vs after return (top 6)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(CHART_DIR / "02_player_rating_timeline.png", dpi=160, bbox_inches="tight")
plt.show()
print("Finding: lines compare mean ratings before injury and after return for the six players with the most paired records; this selected view improves readability and is not a full-player comparison.")
plt.close(fig)

### Chart 3 — Injury frequency by club and month

In [ ]:
# 3. Heatmap of injury counts by club and month
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]
chart_month_data = df.dropna(subset=["Injury_Month", "Team Name"])
chart_heatmap = pd.crosstab(chart_month_data["Team Name"], chart_month_data["Injury_Month"])
chart_heatmap = chart_heatmap.reindex(columns=month_order, fill_value=0)
fig, ax = plt.subplots(figsize=(13, max(5, 0.42 * len(chart_heatmap))))
sns.heatmap(chart_heatmap, cmap="YlOrRd", linewidths=0.25, annot=True,
            fmt="d", cbar_kws={"label": "Recorded injury events"}, ax=ax)
ax.set_title("Injury frequency across months and clubs")
ax.set_xlabel("Month of injury")
ax.set_ylabel("Club")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(CHART_DIR / "03_injury_frequency_club_month_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
if not chart_heatmap.empty and chart_heatmap.to_numpy().max() > 0:
    peak_club, peak_month = np.unravel_index(chart_heatmap.to_numpy().argmax(), chart_heatmap.shape)
    print(f"Finding: the largest club-month cluster is {chart_heatmap.index[peak_club]} in {chart_heatmap.columns[peak_month]} ({int(chart_heatmap.to_numpy().max())} recorded injury events). Counts reflect this dataset's records, not injury rates adjusted for squad size.")

### Chart 4 — Player age versus observed performance-drop index

In [ ]:
# 4. Scatter plot of age against observed performance-drop index
chart_age = df.dropna(subset=["Age", "Performance_Drop_Index"])
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(chart_age["Age"], chart_age["Performance_Drop_Index"], alpha=0.6, edgecolors="none")
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Player age")
ax.set_ylabel("Performance-drop index (before GD − missed-match GD)")
ax.set_title("Player age vs observed performance-drop index")
fig.tight_layout()
fig.savefig(CHART_DIR / "04_age_vs_performance_drop.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
if len(chart_age) >= 3:
    age_corr = chart_age[["Age", "Performance_Drop_Index"]].corr(method="pearson").iloc[0, 1]
    print(f"Finding: Pearson correlation between age and observed performance-drop index is {age_corr:.3f} (n={len(chart_age)} complete records). This describes linear association and does not establish causation.")

### Chart 5 — Comeback leaderboard by mean rating change

In [ ]:
# 5. Comeback leaderboard by average player-rating change
chart_comeback = (
    chart_paired.groupby(["Name", "Team Name"], as_index=False)
    .agg(Injury_Records=("Name", "size"),
         Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean"),
         Mean_Rating_Before=("Avg_Rating_Before", "mean"),
         Mean_Rating_After=("Avg_Rating_After", "mean"))
    .sort_values("Mean_Rating_Change", ascending=False)
)
chart_comeback = chart_comeback[chart_comeback["Injury_Records"] >= 2]
chart_top_comebacks = chart_comeback.head(10).sort_values("Mean_Rating_Change").copy()
chart_top_comebacks["Player_label"] = chart_top_comebacks.apply(lambda r: f'{r["Name"]} (n={int(r["Injury_Records"])})', axis=1)
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(chart_top_comebacks["Player_label"], chart_top_comebacks["Mean_Rating_Change"])
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Mean rating change after return (after − before)")
ax.set_ylabel("Player")
ax.set_title("Comeback leaderboard: mean rating change (minimum 2 paired records)")
fig.tight_layout()
fig.savefig(CHART_DIR / "05_comeback_leaderboard.png", dpi=160, bbox_inches="tight")
plt.show()
if not chart_top_comebacks.empty:
    best_row = chart_comeback.iloc[0]
    print(f"Finding: the largest average rating improvement among players with at least two paired records is {best_row['Name']} ({best_row['Team Name']}), {best_row['Mean_Rating_Change']:.2f} points across {int(best_row['Injury_Records'])} records. This does not establish causation.")
else:
    print("Finding: no players have two paired records; no repeat-record leaderboard can be interpreted.")
plt.close(fig)

### Chart 6 — Recorded injury-to-return duration distribution and long-interval checks

In [ ]:
# 6. Distribution of recorded injury-to-return intervals (includes long intervals/outliers)
chart_duration = df.dropna(subset=["Injury_Duration_Days"])
long_duration_rows = chart_duration[chart_duration["Injury_Duration_Days"] > 365]
print(f"Duration check: {len(long_duration_rows)} intervals exceed 365 days. They are retained and flagged for date/data-entry verification, not automatically removed.")
if not long_duration_rows.empty:
    display(long_duration_rows[["Name", "Team Name", "Injury", "Date of Injury", "Date of return", "Injury_Duration_Days"]].sort_values("Injury_Duration_Days", ascending=False).head(20))
fig, ax = plt.subplots(figsize=(9, 5))
if not chart_duration.empty:
    ax.hist(chart_duration["Injury_Duration_Days"], bins=30)
    ax.set_xlabel("Days from recorded injury date to recorded return date")
    ax.set_ylabel("Number of records")
    ax.set_title("Distribution of recorded injury-to-return intervals")
else:
    ax.text(0.5, 0.5, "No valid injury-to-return intervals", ha="center", va="center")
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(CHART_DIR / "06_injury_duration_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

### Chart 7 — Injury records by club compared with the club average

In [ ]:
# 7. Club injury-count bar chart with an overall-average reference line
# The dashed line makes it easy to identify clubs above/below the mean count per club.
club_counts_chart = (df.groupby("Team Name", as_index=False)
                     .agg(Injury_Records=("Name", "size"))
                     .sort_values("Injury_Records", ascending=True))
mean_club_injuries = club_counts_chart["Injury_Records"].mean() if not club_counts_chart.empty else 0
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(club_counts_chart["Team Name"], club_counts_chart["Injury_Records"])
ax.axvline(mean_club_injuries, color="crimson", linewidth=2, linestyle="--",
           label=f"Average across clubs = {mean_club_injuries:.1f}")
ax.set_xlabel("Recorded injury events")
ax.set_ylabel("Club")
ax.set_title("Injury records by club with overall-average reference line")
ax.legend(loc="lower right")
for y, value in enumerate(club_counts_chart["Injury_Records"]):
    ax.text(value + 1, y, f"{int(value)}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(CHART_DIR / "07_club_injury_counts_average_line.png", dpi=160, bbox_inches="tight")
plt.show()
if not club_counts_chart.empty:
    above = int((club_counts_chart["Injury_Records"] > mean_club_injuries).sum())
    print(f"Finding: the average is {mean_club_injuries:.1f} recorded injury events per club; {above} of {len(club_counts_chart)} clubs are above this dataset-wide average. These are raw counts, not squad-size-adjusted injury rates.")
plt.close(fig)

### Chart 8 — Team goal difference across injury phases

In [ ]:
# 8. Team-by-phase heatmap: mean goal difference before, during absence, and after return
phase_cols = ["Avg_GD_Before", "Avg_GD_Missed", "Avg_GD_After"]
phase_labels = {"Avg_GD_Before": "Before injury", "Avg_GD_Missed": "During absence", "Avg_GD_After": "After return"}
team_phase_heatmap = df.groupby("Team Name")[phase_cols].mean().rename(columns=phase_labels).astype(float)
fig, ax = plt.subplots(figsize=(9, max(4.5, 0.55 * len(team_phase_heatmap))))
sns.heatmap(team_phase_heatmap, cmap="vlag", center=0, annot=True, fmt=".2f",
            linewidths=0.4, cbar_kws={"label": "Mean goal difference"}, ax=ax)
ax.set_title("Mean team goal difference across injury phases")
ax.set_xlabel("Injury phase")
ax.set_ylabel("Club")
fig.tight_layout()
fig.savefig(CHART_DIR / "08_team_performance_phase_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
if not team_phase_heatmap.empty:
    team_phase_change = (team_phase_heatmap["After return"] - team_phase_heatmap["Before injury"]).dropna()
    if not team_phase_change.empty:
        team = team_phase_change.abs().idxmax()
        print(f"Finding: {team} has the largest absolute difference between its mean goal difference before injury and after return ({team_phase_change.loc[team]:+.3f}). Team-level averages are descriptive and may reflect other factors beyond injuries.")

### Chart 9 — Injury-type performance across injury phases

In [ ]:
# 9. Injury-type by phase heatmap for categories with at least five records
eligible_injury_types = (df.dropna(subset=["Injury_Clean"])
                         .groupby("Injury_Clean").size())
eligible_injury_types = eligible_injury_types[eligible_injury_types >= 5].index
phase_by_injury = (df[df["Injury_Clean"].isin(eligible_injury_types)]
                   .groupby("Injury_Clean")[phase_cols].mean()
                   .rename(columns=phase_labels).astype(float))
phase_by_injury = phase_by_injury.drop(index=[x for x in phase_by_injury.index if str(x).casefold() == "quarantine"], errors="ignore")
if not phase_by_injury.empty:
    # Sort by the observed difference between before-injury and missed-match goal difference.
    phase_by_injury = phase_by_injury.assign(_drop=phase_by_injury["Before injury"] - phase_by_injury["During absence"]).sort_values("_drop", ascending=False).drop(columns="_drop").head(15)
    fig, ax = plt.subplots(figsize=(9, max(5, 0.42 * len(phase_by_injury))))
    sns.heatmap(phase_by_injury, cmap="vlag", center=0, annot=True, fmt=".2f",
                linewidths=0.3, cbar_kws={"label": "Mean goal difference"}, ax=ax)
    ax.set_title("Top 15 eligible injury labels: mean goal difference by phase (n ≥ 5 per label)")
    ax.set_xlabel("Injury phase")
    ax.set_ylabel("Injury label")
    fig.tight_layout()
    fig.savefig(CHART_DIR / "09_injury_type_performance_phase_heatmap.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("No injury labels had five records for the injury-type phase heatmap.")

### Additional visualization — performance-drop distribution with a normal reference curve

This chart shows the distribution of the observed performance-drop index and overlays a normal curve using the sample mean and standard deviation. The curve is a **reference for comparison**, not proof that the data are normally distributed.

In [ ]:
# 10. Histogram of performance-drop index with a fitted normal reference curve.
# This is analogous to a binomial histogram + normal curve, but uses the dataset's
# continuous performance-drop measure rather than inventing binomial parameters.
from scipy.stats import norm

drop_values = pd.to_numeric(df["Performance_Drop_Index"], errors="coerce").dropna()
fig, ax = plt.subplots(figsize=(9, 5.5))
if len(drop_values) >= 3 and drop_values.std(ddof=1) > 0:
    mean_drop = drop_values.mean()
    sd_drop = drop_values.std(ddof=1)
    ax.hist(drop_values, bins="auto", density=True, alpha=0.65,
            edgecolor="white", label="Observed distribution")
    x_curve = np.linspace(drop_values.min(), drop_values.max(), 300)
    ax.plot(x_curve, norm.pdf(x_curve, loc=mean_drop, scale=sd_drop),
            linewidth=2, label="Normal reference curve")
    stats_text = (
        f"n = {len(drop_values)}\n"
        f"Mean = {mean_drop:.3f}\n"
        f"SD = {sd_drop:.3f}\n"
        f"Median = {drop_values.median():.3f}"
    )
    ax.text(0.98, 0.96, stats_text, transform=ax.transAxes,
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85))
    ax.set_xlabel("Performance-drop index")
    ax.set_ylabel("Density")
    ax.set_title("Performance-drop distribution with normal reference curve")
    ax.legend()
    print(f"Finding: the performance-drop index has mean {mean_drop:.3f}, median {drop_values.median():.3f}, and standard deviation {sd_drop:.3f} across {len(drop_values)} records.")
    print("Interpretation: the curve is a visual reference only. Assess histogram shape and possible outliers before assuming normality; records may not be independent.")
else:
    ax.text(0.5, 0.5, "Not enough variation or records to fit a normal reference curve.",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(CHART_DIR / "10_performance_drop_distribution_normal_reference.png",
            dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
print(f"Created {len(list(CHART_DIR.glob('*.png')))} chart image files in: {CHART_DIR}")
print("Chart interpretations are printed immediately after the relevant charts; sample counts are included where group averages are compared.")
print("Required rubric visuals shown: injury-impact bar, player-rating timeline, club/month heatmap, age-impact scatter, comeback leaderboard, club-count bar with average line, and two performance-phase heatmaps.")

## 19. Download the cleaned data, summary tables, and charts

Run this cell after the visualization cell. It creates a ZIP containing the engineered dataset, key EDA summary CSVs, and all chart PNGs, including the added average-line bar chart and performance heatmaps. In Google Colab, the ZIP is downloaded through the browser. No second CSV upload is required.

In [ ]:
# Export the cleaned data, summary tables, and chart images, then download one ZIP.
from pathlib import Path
import pandas as pd
import zipfile
import shutil

EXPORT_DIR = Path("player_injuries_eda_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Recalculate export tables from the cleaned dataframe to keep this cell reproducible.
export_tables = {
    "cleaned_engineered_dataset.csv": df,
    "injury_type_counts.csv": (
        df.dropna(subset=["Injury_Clean"])
          .groupby("Injury_Clean", as_index=False)
          .agg(Injury_Records=("Injury_Clean", "size"))
          .sort_values("Injury_Records", ascending=False)
    ),
    "injury_type_impact.csv": (
        df.dropna(subset=["Injury_Clean", "Performance_Drop_Index"])
          .groupby("Injury_Clean", as_index=False)
          .agg(Mean_Performance_Drop=("Performance_Drop_Index", "mean"),
               Median_Performance_Drop=("Performance_Drop_Index", "median"),
               Records=("Performance_Drop_Index", "size"),
               Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean"))
          .query("Records >= 5")
          .loc[lambda t: ~t["Injury_Clean"].str.casefold().isin(["quarantine"])]
          .sort_values("Mean_Performance_Drop", ascending=False)
    ),
    "team_summary.csv": (
        df.groupby("Team Name", as_index=False)
          .agg(Injury_Records=("Name", "size"),
               Unique_Players=("Name", "nunique"),
               Mean_Injury_Duration_Days=("Injury_Duration_Days", "mean"),
               Mean_Performance_Drop_Index=("Performance_Drop_Index", "mean"),
               Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean"))
          .sort_values("Injury_Records", ascending=False)
    ),
    "player_injury_counts.csv": (
        df.groupby(["Name", "Team Name"], as_index=False)
          .agg(Injury_Records=("Injury", "size"),
               Mean_Injury_Duration_Days=("Injury_Duration_Days", "mean"))
          .sort_values("Injury_Records", ascending=False)
    ),
    "player_rating_changes.csv": (
        df.dropna(subset=["Avg_Rating_Before", "Avg_Rating_After"])
          .groupby(["Name", "Team Name"], as_index=False)
          .agg(Injury_Records=("Name", "size"),
               Mean_Rating_Before=("Avg_Rating_Before", "mean"),
               Mean_Rating_After=("Avg_Rating_After", "mean"),
               Mean_Rating_Change=("Rating_Change_After_vs_Before", "mean"))
          .sort_values("Mean_Rating_Change", ascending=False)
    ),
    "monthly_injury_counts.csv": (
        df.dropna(subset=["Injury_Month"])
          .groupby("Injury_Month", as_index=False)
          .size()
          .rename(columns={"size": "Injury_Records"})
    ),
    "team_performance_pivot.csv": pd.pivot_table(
        df, index="Team Name",
        values=["Avg_Rating_Before", "Avg_Rating_After", "Avg_GD_Before",
                "Avg_GD_Missed", "Avg_GD_After"],
        aggfunc="mean"
    ).reset_index()
}

for filename, table in export_tables.items():
    table.to_csv(EXPORT_DIR / filename, index=False)

# Copy generated chart images into the export folder if the visualization cell has run.
chart_source_dir = EXPORT_DIR / "charts"
chart_files = list(chart_source_dir.glob("*.png")) if chart_source_dir.exists() else []
zip_path = Path("player_injuries_eda_outputs.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in EXPORT_DIR.rglob("*"):
        if file_path.is_file() and file_path != zip_path:
            archive.write(file_path, arcname=file_path.relative_to(EXPORT_DIR.parent))

print(f"Prepared {len(export_tables)} CSV tables and {len(chart_files)} chart images.")
print(f"ZIP created: {zip_path.resolve()} ({zip_path.stat().st_size:,} bytes)")

try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("Google Colab download is unavailable in this environment; the ZIP is saved at the path above.")


## 20. Interpretation guide, limitations, and next steps

When writing conclusions, answer each business question directly and quote the relevant table/visual with its sample size. Avoid describing correlation as causation.

**Important limitations**
- The dataset records only a limited set of matches before, during, and after each injury, rather than every club match over a season.
- Goal difference during absence may be affected by opponent strength, venue, team selection, and other injuries; those variables are not controlled here.
- The same player may appear in multiple rows, so injury records are not necessarily independent observations.
- Injury labels may differ in spelling or specificity; this notebook normalizes case/spacing for grouping but does not assume that all similar labels are clinically identical.
- Missing ratings and match outcomes are kept missing, not imputed. Averages use available valid observations, which means sample sizes can vary between phases.
- A recorded return date minus injury date is an approximate interval and may not equal actual days unavailable for matches.

**Recommended follow-up work**
1. Validate the largest apparent injury impacts against match context and sample counts.
2. Compare clubs using rates per squad size or player-season if that information becomes available, rather than raw counts alone.
3. For a dashboard, make team, injury type, season, and month filters available and show sample sizes beside averages.
4. Document the exact meaning and scale of the goal-difference and rating fields in the final submission.

## 21. Rubric alignment and final quality checks

This notebook addresses the EDA requirements in the assignment brief:

- **Problem understanding:** five required business questions are addressed in dedicated sections, with additional questions on repeat injuries, age, and recovery duration.
- **Preprocessing:** missing placeholders, date parsing, numeric conversion, rating/goal-difference feature engineering, injury-duration validation, and readable injury labels are handled explicitly.
- **EDA and insight extraction:** grouped summaries identify injury counts by player and club, injury-type impact, rating changes, recovery intervals, and monthly patterns. A `pivot_table()` compares performance phases.
- **Visual analysis:** the notebook retains interactive Plotly charts and adds six static chart images so the five required visual types remain visible even if HTML rendering fails. The comeback leaderboard is also provided as a ranked table; comparison charts show sample counts and apply minimum record thresholds to reduce small-sample overinterpretation. Long recorded durations are flagged for review rather than automatically deleted.
- **Interpretation discipline:** sample sizes are shown where relevant; missing values are not fabricated; association is not described as causation.

**Data limitation:** the provided CSV contains result labels and goal differences for only a limited set of before/during/after matches. Therefore, results should be described as recorded match observations rather than a complete club win/loss record or causal estimate of injury effects.

Before submission, run all cells from top to bottom in a fresh Colab runtime and confirm that each chart renders. The rubric also separately requires a deployed Streamlit dashboard, GitHub repository, README, and live link; those deliverables are outside the scope of this EDA notebook.